### Lab: Model Evaluation 
#### Anna G. Castillo

##### Step 1: The first step is to create a “Delayed” flag, which will serve as the target attribute. To do this, you
need to find the difference between “ActualDeptTime” and “ScheduledDeptTime” and check if it is greater
than 20 minutes or not. Notice that the departure times are stored in military timestamp format (and
without dates), so a direct subtraction of time values will not work. The suggested data preparation
procedure is as follows:

• First, drop the duplicated entries from the _general.csv data set (subsetting on FlightID).

• Then, merge the two datasets by using ‘FlightID’ as the Primary Key.

• Next, fix the time stamps in the merged data set to include a colon (:) between the hour and the
minute.

• Then, convert these times to the “datetime” format, which is the native format in Python / Pandas
to handle time variables.
   
    o Hint: pd.to_datetime(df['Date'] + ' ' + df['ScheduledTimeWithColons']
    
• Finally, compute the difference between actual and scheduled times (if using datetime format,
the returned values will be in “timedelta” format) and check if this amount is greater than 20
mins. If the value is greater than 20 minutes, label this as “1”, otherwise label it as “0”.
o Hint: (df['actualFull']-df['scheduledFull']) > pd.Timedelta('20 min')


In [1]:
import pandas as pd
import numpy as np

dfGeneral = pd.read_csv('FlightInfo_general.csv', sep=';')
dfTimes = pd.read_csv('FlightInfo_times.csv')

#1
dfGeneral = dfGeneral.drop_duplicates(subset='FlightID')

#2
dfAll = pd.merge(dfGeneral, dfTimes, on='FlightID')

#3
def add_colon(t):
    t = str(t)
    return t[:-2] + ':' + t[-2:]

dfAll['ScheduledTimeWithColons'] = dfAll['ScheduledDeptTime'].apply(add_colon)
dfAll['ActualTimeWithColons'] = dfAll['ActualDeptTime'].apply(add_colon)

#4
dfAll['scheduledFull'] = pd.to_datetime(dfAll['Date'] + ' ' + dfAll['ScheduledTimeWithColons'])
dfAll['actualFull'] = pd.to_datetime(dfAll['Date'] + ' ' + dfAll['ActualTimeWithColons'])

#5
dfAll['delayTime'] = dfAll['actualFull'] - dfAll['scheduledFull']
dfAll['Delayed'] = ((dfAll['actualFull'] - dfAll['scheduledFull']) > pd.Timedelta('20 min')).astype(int)

print(dfAll['Delayed'].value_counts())
dfAll.head()

Delayed
0    1875
1     326
Name: count, dtype: int64


,FlightID,Carrier,Destination,Distance,Date,FlightNumber,Origin,Weather,DayOfWeek,DayOfMonth,TailNumber,ScheduledDeptTime,ActualDeptTime,ScheduledTimeWithColons,ActualTimeWithColons,scheduledFull,actualFull,delayTime,Delayed
0,1,OH,JFK,184,1/1/2004,5935,BWI,0,4,1,N940CA,1455,1455,14:55,14:55,2004-01-01 14:55:00,2004-01-01 14:55:00,0 days 00:00:00,0
1,2,DH,JFK,213,1/1/2004,6155,DCA,0,4,1,N405FJ,1640,1640,16:40,16:40,2004-01-01 16:40:00,2004-01-01 16:40:00,0 days 00:00:00,0
2,3,DH,LGA,229,1/1/2004,7208,IAD,0,4,1,N695BR,1245,1245,12:45,12:45,2004-01-01 12:45:00,2004-01-01 12:45:00,0 days 00:00:00,0
3,4,DH,LGA,229,1/1/2004,7215,IAD,0,4,1,N662BR,1815,1709,18:15,17:09,2004-01-01 18:15:00,2004-01-01 17:09:00,-1 days +22:54:00,0
4,5,DH,LGA,229,1/1/2004,7792,IAD,0,4,1,N698BR,1039,1035,10:39,10:35,2004-01-01 10:39:00,2004-01-01 10:35:00,-1 days +23:56:00,0


##### Step 2: The second step is to create new attributes from the data that can help with prediction. For this
lab session, assume that there is only one new attribute that we’d like to create, ‘TimeOfDay’, as follows:

• ‘TimeOfDay’ is ‘Morning’, if the ‘ScheduledDeptTime’ is between 0:00 and 11:59.

• ‘TimeOfDay’ is ‘Afternoon’, if the ‘ScheduledDeptTime’ is between 12:00 and 17:59.

• ‘TimeOfDay’ is ‘Evening’, if the ‘ScheduledDeptTime’ is between 18:00 and 23:59.

In [4]:
def time_of_day(dt):
    if dt.hour < 12:
        return 'Morning'
    elif dt.hour < 18:
        return 'Afternoon'
    else:
        return 'Evening'

dfAll['TimeOfDay'] = dfAll['scheduledFull'].apply(time_of_day)

print(dfAll['TimeOfDay'].value_counts())

TimeOfDay
Afternoon    1068
Morning       699
Evening       404
Name: count, dtype: int64


##### Step 3: Next, remove all flights with the ‘Career’ code ‘OH’ from the data. After these operations, your
data should look like below:


In [5]:
dfAll = dfAll[dfAll['Carrier'] != 'OH']

print(dfAll.shape)
dfAll.head()

(2171, 20)


,FlightID,Carrier,Destination,Distance,Date,FlightNumber,Origin,Weather,DayOfWeek,DayOfMonth,TailNumber,ScheduledDeptTime,ActualDeptTime,ScheduledTimeWithColons,ActualTimeWithColons,scheduledFull,actualFull,delayTime,Delayed,TimeOfDay
1,2,DH,JFK,213,1/1/2004,6155,DCA,0,4,1,N405FJ,1640,1640,16:40,16:40,2004-01-01 16:40:00,2004-01-01 16:40:00,0 days 00:00:00,0,Afternoon
2,3,DH,LGA,229,1/1/2004,7208,IAD,0,4,1,N695BR,1245,1245,12:45,12:45,2004-01-01 12:45:00,2004-01-01 12:45:00,0 days 00:00:00,0,Afternoon
3,4,DH,LGA,229,1/1/2004,7215,IAD,0,4,1,N662BR,1815,1709,18:15,17:09,2004-01-01 18:15:00,2004-01-01 17:09:00,-1 days +22:54:00,0,Evening
4,5,DH,LGA,229,1/1/2004,7792,IAD,0,4,1,N698BR,1039,1035,10:39,10:35,2004-01-01 10:39:00,2004-01-01 10:35:00,-1 days +23:56:00,0,Morning
5,6,DH,JFK,228,1/1/2004,7800,IAD,0,4,1,N687BR,840,839,8:40,8:39,2004-01-01 08:40:00,2004-01-01 08:39:00,-1 days +23:59:00,0,Morning


##### Step 4: The next step is to prepare the input attributes for the modeling process. Remember that many
scikit-learn techniques only accept numeric values as input. Accordingly, you will need to perform
attribute selection, re-coding, and transformation as follows:

• Not all attributes are meaningful for predicting delay. The attributes that we will use for prediction
are:
['Carrier','Destination','Distance','Origin','Weather','DayOfWeek','DayOfMonth','TimeOfDay'].

• The goal is to convert these attributes to numerical form, so that they can be used with scikitlearn algorithms. Among the attributes, ‘Weather’, ‘Distance’, and ‘DayOfMonth’ are already continuous and numeric, so they can be used as is.

• ‘Career’, ‘Destination’, ‘Origin’, and ‘TimeOfDay’ are categorical and string values, therefore they need to be converted to binary (1/0) form using dummy coding. Another attribute that needs to be treated as categorical is ‘DayOfWeek’. If your data set reads this variable as continuous, you need to, first, convert into categorical form (e.g., using .astype(‘str’)).

• After this, apply dummy coding to encode all categorical attributes using pd.get_dummies function. Make sure to drop one of the levels when dummy coding.

• Once these operations have been completed, compile the final form of these attributes into a new dataframe to be used as the input attribute set (2171 x 26). Below is a sample screenshot:

In [6]:
predictors = ['Carrier', 'Destination', 'Distance', 'Origin',
              'Weather', 'DayOfWeek', 'DayOfMonth', 'TimeOfDay']

dfAll['DayOfWeek'] = dfAll['DayOfWeek'].astype(str)

#Dummy coding of all categorical attributes, dropping one level of each
X = pd.get_dummies(dfAll[predictors], drop_first=True)

y = dfAll['Delayed']

print(X.shape)
X.head()

(2171, 21)


,Distance,Weather,DayOfMonth,Carrier_DH,Carrier_DL,Carrier_MQ,Carrier_RU,Carrier_UA,Carrier_US,Destination_JFK,...,Origin_DCA,Origin_IAD,DayOfWeek_2,DayOfWeek_3,DayOfWeek_4,DayOfWeek_5,DayOfWeek_6,DayOfWeek_7,TimeOfDay_Evening,TimeOfDay_Morning
1,213,0,1,True,False,False,False,False,False,True,...,True,False,False,False,True,False,False,False,False,False
2,229,0,1,True,False,False,False,False,False,False,...,False,True,False,False,True,False,False,False,False,False
3,229,0,1,True,False,False,False,False,False,False,...,False,True,False,False,True,False,False,False,True,False
4,229,0,1,True,False,False,False,False,False,False,...,False,True,False,False,True,False,False,False,False,True
5,228,0,1,True,False,False,False,False,False,True,...,False,True,False,False,True,False,False,False,False,True


##### Step 5: Now, we will build a classification model and evaluate its performance. Let’s use the Neural
Network technique for model learning and both the Holdout and Cross-validation methods for model
evaluation.
Evaluation with Holdout:

• First, make sure that the outcome attribute (class labels) has been separated from the input data set. In other words, you should have an X dataframe with all the input attributes and a y vector with the single output (Delayed) column.

• Split both X and y into “training” and “test” parts, with 70% and 30% proportions, respectively. An easy way to split the data is to use the “train_test_split” function from the scikit-learn’s sklearn.model_selection package. When using this function, make sure that the “random_state” parameter is set to “1” – so that there is consistency among the splits across different runs. For setting the training (test) size, you can use the “train_size” (test_size) parameter. Also note that you need to pass both X and y into the “train_test_split” function at the same time so that their splits are consistent.

• After this operation, you should have four data sets: X_train, X_test, y_train, and y_test.

• Next, build a neural network model (using default parameters and without needing to scale continuous attributes for now) from X_train and y_train and apply the model on X_test. To apply the model, you can call the model’s “.predict” method on “X_test” – which should return the predicted “Delayed” values for the test data set as a vector. Assign this vector to the variable named “predicted_test”.

• Finally, evaluate the performance of the model by comparing “y_test” and “predicted_test”. Assume that our class of interest is “delayed flight” – labeled as “1” in the data. Obtain the overall accuracy as well as the precision and recall scores for the class of interest. Also obtain the confusion matrix.

    o You can use various functions in the sklearn.metrics package for calculating the performance scores such as “classification_report”, “confusion_matrix”, “accuracy_score”, etc.
    
###### Evaluation with Cross-Validation:

• Now, use the cross-validation approach with 5 folds to re-calculate the accuracy, precision, and recall scores. For cross-validation, you need to get the average scores (since this approach will run 5 times). Two helpful functions for this process are: “cross_val_score” and “KFold” in the sklearn.model_selection package.

• When doing cross-validation, a good idea is to instantiate an object using the KFold class first. Use the following parameters for this purpose:
    
    o kfold = KFold(n_splits=5, shuffle=True, random_state=1)

In [7]:
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             confusion_matrix, classification_report)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)

modelNN = MLPClassifier(random_state=1)
modelNN.fit(X_train, y_train)

predicted_test = modelNN.predict(X_test)

acc_holdout = accuracy_score(y_test, predicted_test)
prec_holdout = precision_score(y_test, predicted_test, pos_label=1)
rec_holdout = recall_score(y_test, predicted_test, pos_label=1)

print('Holdout accuracy :', round(acc_holdout, 4))
print('Holdout precision:', round(prec_holdout, 4))
print('Holdout recall   :', round(rec_holdout, 4))

print(confusion_matrix(y_test, predicted_test, labels=[1, 0]))
print(classification_report(y_test, predicted_test))

Holdout accuracy : 0.8313
Holdout precision: 0.7778
Holdout recall   : 0.0609
[[  7 108]
 [  2 535]]
              precision    recall  f1-score   support

           0       0.83      1.00      0.91       537
           1       0.78      0.06      0.11       115

    accuracy                           0.83       652
   macro avg       0.80      0.53      0.51       652
weighted avg       0.82      0.83      0.77       652

